<br>
<font>
<div dir=ltr align=center>
<img src="https://cdn.freebiesupply.com/logos/large/2x/sharif-logo-png-transparent.png" width=150 height=150> <br>
<font color=0F5298 size=7>
    Machine Learning <br>
<font color=2565AE size=5>
    Computer Engineering Department <br>
    Fall 2025<br>
<font color=3C99D size=5>
    Practical Assignment 3 - Deep Learning - Regularization - Neural Network  <br>
<font color=0CBCDF size=4>
   &#x1F335; Amir Malekhosseini  &#x1F335;
</div>

____

# PART 1: Deep Learning Regularization: Dropout, L-Norm, and Batch Normalization

This notebook implements and compares three critical deep learning regularization techniques—Dropout, L2 Regularization, and Batch Normalization—to demonstrate their effectiveness in preventing overfitting on the MNIST dataset, as discussed in the associated article.

In [ ]:
!pip install tensorflow


# 2. Data Preparation (MNIST)

We load the MNIST handwritten digit dataset, flatten the 28x28 images into a 784-dimensional vector, normalize the pixel values to the range $[0, 1]$, and apply one-hot encoding to the labels. We also split the training data to reserve a validation set for monitoring performance and overfitting.

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import numpy as np

# Load MNIST (with fallback)
try:
    (x_train_full, y_train_full), (x_test, y_test) = mnist.load_data()
except Exception:
    from sklearn.datasets import fetch_openml
    mnist_openml = fetch_openml('mnist_784', version=1, as_frame=False)

    X = mnist_openml.data.astype('uint8')
    y = mnist_openml.target.astype('uint8')

    x_train_full = X[:60000].reshape(60000, 28, 28)
    y_train_full = y[:60000]

    x_test = X[60000:].reshape(10000, 28, 28)
    y_test = y[60000:]

# Normalize
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Flatten
x_train_full = x_train_full.reshape(-1, 28 * 28)
x_test = x_test.reshape(-1, 28 * 28)

# One-hot encode labels
NUM_CLASSES = 10
y_train_full = to_categorical(y_train_full, NUM_CLASSES)
y_test = to_categorical(y_test, NUM_CLASSES)

# 80/20 Train/Validation split
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=0.2,
    random_state=14,
    stratify=np.argmax(y_train_full, axis=1)
)

print(f'Training samples: {x_train.shape[0]}')
print(f'Validation samples: {x_val.shape[0]}')
print(f'Test samples: {x_test.shape[0]}')


# 3. Model Definition and Training Functions

We define four models with identical base architecture (two Dense layers with 512 and 256 neurons, and a 10-neuron softmax output layer) but differing regularization strategies:

1.  **Baseline Model:** No regularization.
2.  **Dropout Model:** Uses `layers.Dropout(0.5)` after the hidden layers.
3.  **L2 Regularization Model:** Uses L2 penalty (`regularizers.l2(1e-4)`) on the kernel weights of the dense layers.
4.  **Batch Normalization Model:** Inserts `BatchNormalization()` layers between the dense layer and the activation function.

In [ ]:
INPUT_SHAPE = (784,)
NUM_CLASSES = 10


def create_baseline_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES):
    """
    Creates a simple, unregularized deep network.

    Parameters:
    - input_shape: The shape of the input data (e.g., (784,))
    - num_classes: The number of output classes (e.g., 10)

    Returns:
    - model: A Keras Sequential model instance.
    """
    ###########################################################################
    # Task: Implement a Sequential model without regularization.              #
    # The model should have:                                                  #
    # 1. A Dense layer (512 units, 'relu' activation) with input_shape.       #
    # 2. A Dense layer (256 units, 'relu' activation).                        #
    # 3. A final Dense layer (num_classes, 'softmax' activation).             #
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = models.Sequential(
        [
            layers.Dense(512, activation='relu', input_shape=input_shape),
            layers.Dense(256, activation='relu'),
            layers.Dense(num_classes, activation='softmax'),
        ],
        name='Baseline'
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    return model

def create_dropout_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES, dropout_rate=0.5):
    """
    Creates a model with Dropout regularization.

    Parameters:
    - input_shape: The shape of the input data (e.g., (784,))
    - num_classes: The number of output classes (e.g., 10)
    - dropout_rate: The rate of dropout (e.g., 0.5)

    Returns:
    - model: A Keras Sequential model instance with Dropout layers.
    """
    ###########################################################################
    # Task: Implement a Sequential model with Dropout Regularization.         #
    # Insert a Dropout layer after *each* hidden Dense layer.                 #
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = models.Sequential(
        [
            layers.Dense(512, activation='relu', input_shape=input_shape),
            layers.Dropout(dropout_rate),
            layers.Dense(256, activation='relu'),
            layers.Dropout(dropout_rate),
            layers.Dense(num_classes, activation='softmax'),
        ],
        name='Dropout'
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    return model


def create_l2_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES, l2_factor=1e-4):
    """
    Creates a model with L2 Weight Regularization.

    Parameters:
    - input_shape: The shape of the input data (e.g., (784,))
    - num_classes: The number of output classes (e.g., 10)
    - l2_factor: The L2 regularization factor.

    Returns:
    - model: A Keras Sequential model instance with L2 regularization.
    """
    l2_reg = regularizers.l2(l2_factor)
    ###########################################################################
    # Task: Implement a Sequential model with L2 Weight Regularization.       #
    # Apply L2 regularization (using the provided l2_reg object) to the       #
    # 'kernel_regularizer' argument of *both* hidden Dense layers.            #
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = models.Sequential(
        [
            layers.Dense(512, activation='relu', input_shape=input_shape, kernel_regularizer=l2_reg),
            layers.Dense(256, activation='relu', kernel_regularizer=l2_reg),
            layers.Dense(num_classes, activation='softmax'),
        ],
        name='L2'
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    return model


def create_batch_norm_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES):
    """
    Creates a model with Batch Normalization layers.

    Parameters:
    - input_shape: The shape of the input data (e.g., (784,))
    - num_classes: The number of output classes (e.g., 10)

    Returns:
    - model: A Keras Sequential model instance with Batch Normalization layers.
    """
    ###########################################################################
    # Task: Implement a Sequential model with Batch Normalization (BN).       #
    # Insert a BN layer after *each* hidden Dense layer, and ensure the       #
    # activation is applied *after* the BN layer.                             #
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = models.Sequential(
        [
            layers.Dense(512, input_shape=input_shape),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.Dense(256),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.Dense(num_classes, activation='softmax'),
        ],
        name='BatchNorm'
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    return model


# The following functions and execution block are left intact, as they handle training and evaluation.
# You may want to comment out the final execution block (as shown in my first response)
# if you want students to also write the model instantiation, training, and evaluation loops.

def compile_and_train(model, x_train, y_train, x_val, y_val,
                      epochs=20, batch_size=128):
    """
    Compiles and trains a given model and returns the history.
    """
    print(f'\n--- Training {model.name} Model ---\n')
    ###########################################################################
    # Task 5: Implement the compilation and training steps.                   #
    # 1. Compile the model using 'adam' optimizer and 'categorical_crossentropy'.
    # 2. Train the model using the provided x/y_train/val data and parameters.
    # 3. Return the training history object.
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    return history



# Create and store models (Assuming x_train, y_train, etc. are loaded elsewhere)
models_dict = {
    'Baseline': create_baseline_model(),
    'Dropout (0.5)': create_dropout_model(),
    'L2 Regularization (1e-4)': create_l2_model(),
    'Batch Normalization': create_batch_norm_model()
}

histories = {}

# Train all models
for model_name, model in models_dict.items():
    history = compile_and_train(model, x_train, y_train, x_val, y_val)
    histories[model_name] = history

# Evaluate all models on the test set
test_results = {}
for model_name, model in models_dict.items():
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    test_results[model_name] = {'loss': test_loss, 'accuracy': test_acc}

    print(f'{model_name} Test Accuracy: {test_acc:.4f}, Test Loss: {test_loss:.4f}')

# 4. Visualization and Comparison

The most important part of regularization is observing the gap between training and validation performance. A good regularizer should minimize this gap and achieve a high test accuracy.

In [ ]:
def plot_history(histories, key='accuracy'):
    """
    Plots training and validation metrics for all models.

    Parameters:
    - histories: A dictionary where keys are model names (str) and values are
                 Keras History objects returned from model.fit().
    - key: The metric to plot ('accuracy' or 'loss').
    """
    plt.figure(figsize=(14, 6))

    # Define line styles and markers for clarity
    styles = ['-', '--', ':', '-.']
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    i = 0
    for name, history in histories.items():
        ###########################################################################
        # 1. Initialize an index 'i = 0'.                                         #
        # 2. Loop through the 'histories' dictionary.                             #
        # 3. Inside the loop, determine the line_style and color using 'i'.       #
        # 4. Plot the Training metric (key) and the Validation metric (f'val_{key}').
        # 5. Increment the index 'i'.                                             #
        ###########################################################################

        line_style = styles[i % len(styles)]
        color = colors[i % len(colors)]

        plt.plot(history.history[key], linestyle=line_style, color=color, label=f'{name} Train')
        plt.plot(history.history[f'val_{key}'], linestyle=line_style, color=color, label=f'{name} Val', alpha=0.75)

        i += 1

    plt.title(
        f'Training and Validation {key.capitalize()} Comparison', fontsize=16)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel(key.capitalize(), fontsize=12)
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# Plot Accuracy
plot_history(histories, key='accuracy')

# Plot Loss
plot_history(histories, key='loss')

# Questions (Part 1)

**Question 1:** Both L2 Regularization and Dropout are designed to prevent overfitting. Explain the fundamental difference in how each technique achieves this goal, and describe the potential trade-off you must manage when tuning the L2 weight λ or the Dropout rate p.


Both L2 Regularization and Dropout aim to reduce overfitting, but they achieve this goal through fundamentally different mechanisms.

L2 Regularization adds a penalty term to the loss function that is proportional to the squared magnitude of the model weights. This encourages the network to learn smaller weight values, effectively limiting model complexity and preventing it from fitting noise in the training data. The regularization is applied continuously during training and directly constrains the parameter space.

In contrast, Dropout randomly deactivates a fraction of neurons during each training iteration with probability 
𝑝
p. This forces the network to learn redundant and more robust feature representations and prevents excessive reliance on any single neuron. Dropout can be interpreted as training an ensemble of many sub-networks and averaging their predictions.

Trade-off:
If the L2 regularization coefficient 
𝜆
λ is set too high, the model may become overly constrained and underfit the data. Similarly, if the Dropout rate 
𝑝
p is too large, the effective capacity of the network is reduced, which can slow convergence or harm final performance. Therefore, careful tuning is required to balance regularization strength and model expressiveness.



**Question 2:** In the create_batch_norm_model, a Batch Normalization (BN) layer is placed before the activation function. If you were to place the BN layer after the ReLU activation, what practical impact might that have on the model's training dynamics or final performance, and why?


Placing Batch Normalization (BN) before the activation function ensures that the inputs to the activation have a normalized distribution with controlled mean and variance. This reduces internal covariate shift, stabilizes gradients, and generally leads to faster and more reliable training.

If Batch Normalization is applied after the ReLU activation, the normalization is performed on a distribution that is already truncated at zero. This can weaken the sparsity-inducing effect of ReLU and lead to less stable activation distributions. As a result, training dynamics may become slightly less efficient, and final performance can degrade.

In practice, placing Batch Normalization before the activation function typically results in faster convergence, improved training stability, and better generalization, which is why it is the standard design choice in most modern neural network architectures.


**Question 3:** Based on the performance plots you generated (comparing the baseline, Dropout, L2, and BN models), which model showed the largest gap between its training loss and validation loss? What does this specific result tell you about the problem of overfitting in that particular model?

Based on the training and validation performance plots, the Baseline model without any regularization exhibits the largest gap between training loss and validation loss. While the training loss decreases rapidly, the validation loss stops improving or even increases after a few epochs.

This large gap indicates that the baseline model is memorizing the training data rather than learning generalizable patterns. In other words, the model is suffering from overfitting. The regularized models (Dropout, L2 Regularization, and Batch Normalization) show smaller gaps between training and validation loss, demonstrating improved generalization to unseen data.

# PART 2:  The MNIST Dataset Challenge

> **Goal:** Build a machine learning model that achieves an accuracy of **97% or higher** on the famous **MNIST dataset**. This dataset is a standard benchmark containing images of handwritten digits (0-9). Meeting this high accuracy threshold requires careful design, training, and evaluation of your model.

---

### Task Breakdown

Using only the provided libraries, you must complete the following four key steps:

* **Step 1: Data Preparation.** You need to **load and preprocess** the raw MNIST data to make it suitable for training a neural network.
* **Step 2: Model Architecture.** Your task is to **build** a robust neural network model specifically designed for image classification.
* **Step 3: Training & Tuning.** You will **train** the model on the training data. This involves monitoring the validation accuracy and carefully adjusting hyperparameters (parameters) to optimize performance.
* **Step 4: Final Assessment.** Finally, you must **evaluate** the model on the test data to formally ensure that the required 97% or higher accuracy threshold has been met.

---

**Instruction:** Each subsequent cell in this notebook is designed to guide you through these steps sequentially. Read the descriptions carefully, and apply your knowledge of neural networks to achieve the best possible performance on this classification problem.

###  Data Preparation: Step 1 Details

This notebook section focuses entirely on **preparing the MNIST dataset** so that it is suitable for input into our neural network model.

---

The successful completion of this cell requires four distinct actions:

| **Action** | **Purpose and Requirement** |
| :--- | :--- |
| **Load the Raw Data** | The MNIST dataset contains 28x28 grayscale images of digits (0–9). Load both the image arrays (pixel values) and their corresponding labels (the digit they represent). |
| **Set Up Constants** | You must define three key variables for model architecture and dimension checks: `num_classes` (the number of unique digits, 0–9), `input_shape` (the 28x28 image dimensions), and `num_features` (the total number of pixels, 784, if flattening is necessary). |
| **Normalize Images** | Rescale the pixel values by dividing them by **255**. This crucial step transforms the initial range of 0–255 to the normalized range of **0–1**, which significantly aids model convergence and overall performance. |
| **Encode Target Labels** | Convert the integer labels into a **one-hot encoded** format. For instance, the integer label `3` must become a binary vector like `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`. This structure is mandatory for training models on multi-class classification tasks. |

---

> By completing these steps, the dataset will be cleaned, scaled, and formatted, making the data entirely ready for the subsequent model building and training phases.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from keras_tuner import Hyperband, RandomSearch, BayesianOptimization

from tensorflow import argmax
from tensorflow.keras.datasets import mnist
from tensorflow.math import confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import SGD, Adagrad, RMSprop, Adam

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np

# Load MNIST (with fallback, identical output)
try:
    (x_train, y_train), (x_test, y_test) = mnist.load_data()
except Exception:
    from sklearn.datasets import fetch_openml

    mnist_openml = fetch_openml('mnist_784', version=1, as_frame=False)

    X = mnist_openml.data.astype('uint8')
    y = mnist_openml.target.astype('uint8')

    x_train = X[:60000].reshape(60000, 28, 28)
    y_train = y[:60000]

    x_test = X[60000:].reshape(10000, 28, 28)
    y_test = y[60000:]

# Constants
num_classes = len(np.unique(y_train))
input_shape = x_train.shape[1:]
num_features = np.prod(input_shape)

# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# One-hot encode labels
y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)


##  Model Construction & Optimization (Step 2)

The core objective of this cell is to define and refine a neural network to secure an accuracy of **97% or higher** on the MNIST test set.

###  Key Optimization Actions

To achieve the required performance, you must execute the following three critical tasks, leveraging the best practices for deep learning architecture design and tuning:

1.  **Architecture Definition**: Design the core **neural network architecture**. Experimentation is key—vary the number of layers, the unit count per layer, and select appropriate activation functions to maximize performance.
2.  **Optimizer Selection**: Test and compare various **optimizers** (e.g., *Adam*, *RMSprop*, *SGD*). Crucially, fine-tune the associated hyperparameters, such as the initial learning rate ($\alpha$) or momentum, for the chosen optimizer.
3.  **Hyperparameter Search**: Employ an efficient hyperparameter tuning strategy to locate the optimal combination of settings.

###  Recommended Search Methods

Consider utilizing one of these advanced search techniques for efficiency:

* **Hyperband**: Best for rapid iteration; it quickly filters out poorly performing configurations.
* **Random Search**: A reliable baseline; it explores a wide, defined range of parameter combinations.
* **Bayesian Optimization**: An intelligent approach that leverages past results to guide the search for better parameters.

###  Final Deliverables

Upon successfully reaching the **>97% accuracy** milestone on the MNIST test data, conclude this cell by providing three distinct outputs:

1.  **Optimal Parameters**: Clearly **print the optimal hyperparameters** identified through your tuning process.
2.  **Final Model Build**: **Build and compile the final model** using these newly identified optimal parameters.
3.  **Summary Display**: **Display the complete model summary** (i.e., the table of layers, output shapes, and parameter counts).

This disciplined approach ensures you fully understand the impact of configuration choices on model efficacy.

In [ ]:
def model_creator(optimizer='adagrad', learning_rate=0.001, num_units=128):
    """
    Creates and compiles a simple feedforward neural network for MNIST-like classification.

    Parameters:
    - optimizer (str): Name of the optimizer to use ('adagrad', 'rmsprop', 'adam', or 'sgd')
    - learning_rate (float): Learning rate for the optimizer
    - num_units (int): Number of units in each hidden layer

    Returns:
    - model (tf.keras.Sequential): Compiled Keras model
    """
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(num_units, activation='relu'),
        Dense(num_units, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])

    ###########################################################################
    # Task: Create the appropriate optimizer instance based on the `optimizer` #
    #       argument. Use the provided `learning_rate`. Supported optimizers:  #
    #       'adagrad', 'rmsprop', 'adam', and 'sgd'.                          #
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    opt_name = optimizer.lower()
    if opt_name == 'adagrad':
        optimizer = Adagrad(learning_rate=learning_rate)
    elif opt_name == 'rmsprop':
        optimizer = RMSprop(learning_rate=learning_rate)
    elif opt_name == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif opt_name == 'sgd':
        optimizer = SGD(learning_rate=learning_rate)
    else:
        raise ValueError("Unsupported optimizer. Choose from: 'adagrad', 'rmsprop', 'adam', 'sgd'.")

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy', metrics=['accuracy'])

    return model

    ###########################################################################
    # Task: Create the appropriate optimizer instance based on the `optimizer` #
    #       argument. Use the provided `learning_rate`. Supported optimizers:  #
    #       'adagrad', 'rmsprop', 'adam', and 'sgd'.                          #
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****




def hyperparameters_model_creator(hyper_parameter):
    """
    Defines the hyperparameter search space and returns a model built with those hyperparameters.

    Parameters:
    - hyper_parameter: A KerasTuner HyperParameters object used to define the search space.

    Returns:
    - hyper_model: A compiled Keras model using hyperparameters sampled from the defined space.
    """

    ###########################################################################
    # Task: Define three hyperparameters using the `hyper_parameter` object:   #
    #   1. 'optimizer': a Choice among ['rmsprop', 'sgd', 'adam', 'adagrad']   #
    #   2. 'learning_rate': a Float from 1e-5 to 1e-1 (log-sampled)           #
    #   3. 'num_units': an Int from 32 to 512 in steps of 32                   #
    # Then, pass them to `model_creator` to build the model.                  #
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = hyper_parameter.Choice('optimizer', ['rmsprop', 'sgd', 'adam', 'adagrad'])
    learning_rate = hyper_parameter.Float('learning_rate', min_value=1e-5, max_value=1e-1, sampling='log')
    num_units = hyper_parameter.Int('num_units', min_value=32, max_value=512, step=32)

    hyper_model = model_creator(
        optimizer=optimizer,
        learning_rate=learning_rate,
        num_units=num_units
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    return hyper_model

    ###########################################################################
    # Task: Define three hyperparameters using the `hyper_parameter` object:   #
    #   1. 'optimizer': a Choice among ['rmsprop', 'sgd', 'adam', 'adagrad']   #
    #   2. 'learning_rate': a Float from 1e-5 to 1e-1 (log-sampled)           #
    #   3. 'num_units': an Int from 32 to 512 in steps of 32                   #
    # Then, pass them to `model_creator` to build the model.                  #
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****




In [ ]:
from tensorflow.keras.callbacks import Callback


class StopAtAccuracy(Callback):
    def __init__(self, accuracy_threshold):
        super(StopAtAccuracy, self).__init__()
        self.accuracy_threshold = accuracy_threshold

    def on_epoch_end(self, epoch, logs=None):
        val_accuracy = (logs or {}).get("val_accuracy")
        if val_accuracy and val_accuracy >= self.accuracy_threshold:
            print(
                f"\nStopping training as validation accuracy reached {val_accuracy:.2f}")
            self.model.stop_training = True

# You may assume `x_train`, `y_train`, `x_test`, `y_test`, and `hyperparameters_model_creator`
# are already defined in the notebook environment.

###########################################################################
# Task: Set up and run a KerasTuner Hyperband search using the following  #
#       specifications:                                                   #
#   - Use `hyperparameters_model_creator` as the model-building function  #
#   - Optimize for 'accuracy'                                             #
#   - Max 5 epochs per trial, factor=3                                    #
#   - Directory: 'my_dir', Project name: 'hyperparameter_tuning'          #
#   - Search using:                                                       #
#         x_train, y_train for training                                   #
#         (x_test, y_test) for validation                                 #
#         callbacks=[stop_at_accuracy]                                    #
#         verbose=1                                                       #
#   - Define `accuracy_threshold = 0.97` and create the callback instance  #
#   - After tuning, retrieve the best model and best hyperparameters       #
#   - Print the best 'num_units', 'optimizer', and 'learning_rate'         #
#   - Call `.summary()` on the best model                                 #
###########################################################################
# *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

accuracy_threshold = 0.97
stop_at_accuracy = StopAtAccuracy(accuracy_threshold)

tuner = Hyperband(
    hyperparameters_model_creator,
    objective='accuracy',
    max_epochs=5,
    factor=3,
    directory='my_dir',
    project_name='hyperparameter_tuning'
)

tuner.search(
    x_train, y_train,
    validation_data=(x_test, y_test),
    callbacks=[stop_at_accuracy],
    verbose=1
)

best_model = tuner.get_best_models(num_models=1)[0]
best_hyperparameters = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best num_units:", best_hyperparameters.get('num_units'))
print("Best optimizer:", best_hyperparameters.get('optimizer'))
print("Best learning_rate:", best_hyperparameters.get('learning_rate'))

best_model.summary()

model = best_model

# *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****


##  Model Operational Phase (Step 3)

The objective of this final section is to execute the training process, formally assess the model's performance against the benchmark, and ensure the trained weights are permanently stored.

###  Required Actions (Sequential)

Complete the following four mandatory tasks in sequence:

1.  **Initiate Training**: Call the **`fit`** function to begin training the model on the MNIST training data. Select an appropriate number of epochs and a batch size. You must monitor the output to track the model's improvement in both training and validation accuracy/loss across each epoch.
2.  **Generate Test Predictions**: Using the trained model, **generate predictions** for the entire MNIST test dataset. (Although not strictly required for the final evaluation metric, having the raw predictions is vital for potential post-analysis, such as generating a confusion matrix.)
3.  **Formal Evaluation**: Utilize the **`evaluate`** function to formally compute the final loss and, most importantly, the **accuracy** on the reserved test set. This is the crucial step to verify that the target performance—**97% or higher** accuracy—has been met.
4.  **Model Persistence**: Once the accuracy target is confirmed, **save the entire trained model** to disk. This ensures that the architecture, configuration, and trained weights are preserved for future deployment.

---

### Configuration and Storage Information

* **Adjustments**: The parameters within the `fit` function, specifically the **number of epochs** or the **batch size**, can be dynamically adjusted here to aid the model in reaching the **$\geq 97\%$** accuracy target.
* **Output File**: The model will be saved to a file named **`mnist_model.keras`**. This single file encapsulates the full model structure and its learned weights, making future loading and deployment straightforward and eliminating the need for retraining.

By the end of this cell, a high-performance model will be trained, evaluated, and permanently archived. 

In [ ]:
# Assume `model`, `x_train`, `y_train`, `x_test`, and `y_test` are already defined.

###########################################################################
# Task: Complete the following steps using the trained `model`:            #
#   1. Train the model for 10 epochs with batch_size=32 and               #
#      validation_data=(x_test, y_test). Store the result in `history`.    #
#   2. Generate predictions on the test set (`x_test`).                    #
#   3. Evaluate the model on the test set to obtain `loss` and `accuracy`. #
#   4. Save the model to a file named 'mnist_model.keras'.                 #
#                                                                        #
# The print statements below are provided—just make sure `loss` and       #
# `accuracy` are defined correctly so they work as expected.              #
###########################################################################
# *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(x_test, y_test),
    verbose=1
)

predictions = model.predict(x_test, verbose=0)

loss, accuracy = model.evaluate(x_test, y_test, verbose=0)

# *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****


print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy*100:.2f}%")

model.save('mnist_model.keras')

##  Final Analysis: Visualizing Model Performance (Step 4)

This concluding step is dedicated to the visual interpretation of your model's training process and final classification results. These visualizations are essential for diagnosing performance and informing future improvements.

---

### **Primary Visualization Tasks**

You are required to generate and analyze the following three critical plots:

* **1. Training History Plot:**
    * **Content:** Display the **training accuracy** and **validation accuracy** across every epoch.
    * **Purpose:** This reveals the learning progression. Use it to check for **overfitting**—a significant gap where training accuracy far exceeds validation accuracy—or if the model converged effectively.

* **2. Confusion Matrix Plot:**
    * **Content:** A matrix showing the true versus predicted classifications for all ten digit classes ($\text{0–9}$).
    * **Purpose:** Provides a **detailed, class-by-class breakdown** of performance. Non-zero values on the off-diagonal cells highlight specific pairs of digits (e.g., *4* and *9*) that the model most frequently misclassifies.

* **3. Misclassified Image Grid:**
    * **Content:** A grid displaying a selection of the images that the model classified **incorrectly**. Each image must be labeled with its **true label** and the model's **predicted label**.
    * **Purpose:** Offers direct **qualitative insight** into model errors. Examining these visually similar but incorrectly predicted examples (e.g., a handwritten *3* predicted as a *5*) guides further model adjustments.

---

### ** Interpretation Guidance**

**Model Insight** | **Visual Clue** | **Actionable Tip**
| :--- | :--- | :--- |
| **Learning Stability** | Validation accuracy flatlines or drops early. | Consider adjusting the learning rate or increasing regularization. |
| **Classification Errors** | High numbers in off-diagonal cells of the Confusion Matrix. | Focus on tuning the model for those specific, struggling digit classes. |
| **Data Challenges** | Misclassified images share ambiguous features. | The current model may need more complex architecture or better image pre-processing. |

By meticulously analyzing these plots, you gain a deep understanding of your model's strengths and **weaknesses** on the MNIST challenge.

In [ ]:
# Plot the training history
plt.figure(figsize=(10, 6))
plt.plot(history.history['accuracy'], label='train_accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training History')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Assume `predictions` and `y_test` are already defined (from model.predict and test labels).

###########################################################################
# Task: Generate and plot a confusion matrix heatmap using the test       #
#       predictions and true labels.                                      #
#   - Convert `predictions` and `y_test` from one-hot to class indices    #
#     using np.argmax (axis=1).                                           #
#   - Compute the confusion matrix using `confusion_matrix`.              #
#   - Plot it with `sns.heatmap` in a figure of size (8, 6), with         #
#     annotations (`annot=True`) and integer formatting (`fmt='d'`).      #
#                                                                        #
# The axis labels and title below are provided—just make sure `cm` is     #
# correctly defined so the plot works.                                    #
###########################################################################
# *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

y_pred = np.argmax(predictions, axis=1)
y_true = np.argmax(y_test, axis=1)
cm = confusion_matrix(y_true, y_pred).numpy()

# *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****


plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Assume `x_test`, `y_test`, and `predictions` are already defined.

###########################################################################
# Task: Find the images that were incorrectly classified by the model and #
#       display the first 20 of them in a 4x5 grid. For each image, show  #
#       the true label and the predicted label in the title.              #
###########################################################################
# *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

y_pred = np.argmax(predictions, axis=1)
y_true = np.argmax(y_test, axis=1)

mis_idx = np.where(y_pred != y_true)[0]
mis_idx = mis_idx[:20]

plt.figure(figsize=(12, 10))
for i, idx in enumerate(mis_idx, start=1):
    plt.subplot(4, 5, i)
    plt.imshow(x_test[idx], cmap='gray')
    plt.title(f"True: {y_true[idx]} | Pred: {y_pred[idx]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

# *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****


# Questions (Part 2)

**Question 4:** In Part 1, you found that one of the regularized models performed best (e.g., the Dropout model). Explain why the Test Accuracy (or Test Loss) is a better metric than the Validation Accuracy (or Validation Loss) for deciding which model to use for the final error analysis in Part 2.



The test accuracy (or test loss) is a more reliable metric than validation accuracy when selecting the final model for error analysis because the test set represents completely unseen data that played no role in either training or model selection.

The validation set is used repeatedly during model development for hyperparameter tuning, architectural decisions, and early stopping. As a result, the model selection process may indirectly adapt to the validation data, causing a slight optimistic bias in validation performance.

In contrast, the test set is evaluated only once after all design decisions are finalized. Therefore, test accuracy provides an unbiased estimate of the model’s true generalization performance. For this reason, conclusions drawn from error analysis (such as confusion matrices or misclassified examples) are more trustworthy when they are based on test results rather than validation results.








**Question 5:** Assume the Dropout Model and the Batch Normalization (BN) Model achieved nearly identical Test Accuracy (e.g., 98.05% vs. 98.07%). If your error analysis from Part 2 revealed that the BN model misclassified digits that were visually clear, while the Dropout model's errors were only on extremely ambiguous digits, which model would you choose for deployment and why? Justify your answer based on the concepts of model robustness and generalization.






In this scenario, the Dropout model should be preferred for deployment.

Although the Dropout and Batch Normalization models achieve nearly identical test accuracy, error analysis reveals a crucial qualitative difference: the Batch Normalization model makes errors on visually clear digits, while the Dropout model’s mistakes are limited to highly ambiguous samples.

This indicates that the Dropout model has learned more robust and semantically meaningful representations, as it correctly classifies clear patterns and only fails on inherently difficult cases. Such behavior reflects stronger generalization and better resistance to unexpected or noisy inputs.

From a deployment perspective, robustness is often more important than marginal differences in aggregate accuracy. A model that fails on obvious cases may behave unpredictably in real-world conditions, whereas a model whose errors are confined to ambiguous inputs is more reliable. Therefore, despite similar numerical performance, the Dropout model is the better choice for deployment.
